In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [2]:
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer, TorchAoConfig
from torchao.quantization.quant_api import Int4WeightOnlyConfig
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer, setup_chat_format
import torch
import os
from dotenv import load_dotenv
from peft import LoraConfig, get_peft_model, AutoPeftModelForCausalLM
import copy
load_dotenv()

# Load a sample dataset
from datasets import load_dataset

W0618 23:23:02.248000 12146 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


True

In [3]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Using device: {device}")

Using device: mps


In [4]:
# Load the model and tokenizer
# model_name = "HuggingFaceTB/SmolLM2-135M"
model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

# Create quantization configuration
quantization_config = TorchAoConfig(quant_type="int8_weight_only")


In [6]:

# Load and automatically quantize
model = AutoModelForCausalLM.from_pretrained(
   model_name,
   # device_map="auto",
   # torch_dtype=torch.float16,
   quantization_config=quantization_config
).to(device)


# model = AutoModelForCausalLM.from_pretrained(model_name, 
#                                              quantization_config=eetq_config).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name, )

if tokenizer.chat_template is None:
   model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)


In [7]:
prompt = "Robots are becoming more and more common in our daily lives. \
Can you tell me how they are being used today?"

formatted_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=False
)
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

In [8]:
default_model = copy.deepcopy(model)

In [9]:
outputs = default_model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.2,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Robots are becoming more and more common in our daily lives. Can you tell me how they are being used today?<|im_end|>
<|im_start|>assistant
Absolutely, robots are becoming increasingly common in our daily lives. They are being used in various sectors such as healthcare, manufacturing, education, and more. Here are some examples:

1. Healthcare: Robots are being used in hospitals to perform tasks such as surgery, blood drawing, and wound care. They are also being used in medical research to analyze data and develop new treatments.

2. Manufacturing: Robots are being used in manufacturing to perform tasks such as assembly line production


In [10]:
# TODO: Configure LoRA parameters
# r: rank dimension for LoRA update matrices (smaller = more compression)
rank_dimension = 8
# lora_alpha: scaling factor for LoRA layers (higher = stronger adaptation)
lora_alpha = 16
# lora_dropout: dropout probability for LoRA layers (helps prevent overfitting)
lora_dropout = 0.1

peft_config = LoraConfig(
    r=rank_dimension,  # Rank dimension - typically between 4-32
    lora_alpha=lora_alpha,  # LoRA scaling factor - typically 2x rank
    lora_dropout=lora_dropout,  # Dropout probability for LoRA layers
    bias="none",  # Bias type for LoRA. the corresponding biases will be updated during training.
    target_modules="all-linear",  # Which modules to apply LoRA to
    task_type="CAUSAL_LM",  # Task type for model architecture
    
)
peft_model = get_peft_model(model, peft_config)
peft_model = peft_model.to(device)

peft_model.print_trainable_parameters()

/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'
trainable params: 2,442,240 || all params: 136,957,248 || trainable%: 1.7832


In [11]:
# Use a reasoning dataset
ds = load_dataset("prithivMLmods/Deepthink-Reasoning")

In [12]:
def tokenize_function(examples):
    prompts = [p.strip() for p in examples["prompt"]]
    responses = [r.strip() for r in examples["response"]]
    texts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": p}, {"role": "assistant", "content": r}],
            tokenize=False
        )
        for p, r in zip(prompts, responses)
    ]
    return tokenizer(texts, truncation=True, padding="max_length", max_length=512, )

ds = ds.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing dataset",
)

Tokenizing dataset:   0%|          | 0/251 [00:00<?, ? examples/s]

In [13]:
finetune_name = "SmolLM2-FT-MyDataset"
# Training configuration
# Hyperparameters based on QLoRA paper recommendations

# Configure the SFTTrainer
sft_config = SFTConfig(
    output_dir=finetune_name,
    max_steps=400,  # Adjust based on dataset size and desired training duration
    per_device_train_batch_size=8,  # Set according to your GPU memory capacity
    logging_steps=20,  # Frequency of logging training metrics
    save_steps=100,  # Frequency of saving model checkpoints
    # eval_strategy="steps",  # Evaluate the model at regular intervals
    # eval_steps=50,  # Frequency of evaluation
    use_mps_device=(
        True if device == "mps" else False
    ),  # Use MPS for mixed precision training
    hub_model_id=finetune_name,  # Set a unique name for your model
    optim="adamw_torch_fused",  # Use fused AdamW for efficiency
    learning_rate=2e-4,  # Learning rate (QLoRA paper)
    max_grad_norm=0.3,  # Gradient clipping threshold
    # Learning rate schedule
    warmup_ratio=0.03,  # Portion of steps for warmup
    lr_scheduler_type="constant",  # Keep learning rate constant after warmup
)


# args = SFTConfig(
#     # Output settings
#     output_dir=finetune_name,  # Directory to save model checkpoints
#     # Training duration
#     max_steps=400,  # Adjust based on dataset size and desired training duration
#     per_device_train_batch_size=4,  # Set according to your GPU memory capacity
#     logging_steps=20,  # Frequency of logging training metrics
#     save_steps=100,  # Frequency of saving model checkpoints
#     # Batch size settings
#     gradient_accumulation_steps=2,  # Accumulate gradients for larger effective batch
#     # Memory optimization
#     gradient_checkpointing=True,  # Trade compute for memory savings
#     # Optimizer settings
#     optim="adamw_torch_fused",  # Use fused AdamW for efficiency
#     learning_rate=2e-4,  # Learning rate (QLoRA paper)
#     max_grad_norm=0.3,  # Gradient clipping threshold
#     # Learning rate schedule
#     warmup_ratio=0.03,  # Portion of steps for warmup
#     lr_scheduler_type="constant",  # Keep learning rate constant after warmup
#     # Logging and saving
#     save_strategy="epoch",  # Save checkpoint every epoch
#     # Precision settings
#     # bf16=True,  # Use bfloat16 precision
#     # Integration settings
#     push_to_hub=False,  # Don't push to HuggingFace Hub
#     report_to="none",  # Disable external logging
#     use_mps_device=(
#         True if device == "mps" else False
#     ),
# )


# Create SFTTrainer with LoRA configuration
trainer = SFTTrainer(
    model=peft_model,
    args=sft_config,
    train_dataset=ds["train"],
    peft_config=peft_config,  # LoRA configuration

)

/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/transformers/training_args.py:2214: UserWarning: `use_mps_device` is deprecated and will be removed in version 5.0 of 🤗 Transformers. `mps` device will be used by default if available similar to the way `cuda` device is used.Therefore, no action from user is required. 
  warnings.warn(


Truncating train dataset:   0%|          | 0/251 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [14]:
trainer.train()

/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
20,1.392600
40,1.107200
60,0.997300
80,0.891200
100,0.947000
120,0.906200
140,0.869400
160,0.823900
180,0.816100
200,0.798200


/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=400, training_loss=0.8174688172340393, metrics={'train_runtime': 674.1341, 'train_samples_per_second': 4.747, 'train_steps_per_second': 0.593, 'total_flos': 1048005075271680.0, 'train_loss': 0.8174688172340393})

In [15]:
# Load PEFT model on CPU
model = AutoPeftModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=sft_config.output_dir + "/checkpoint-400",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

# Merge LoRA and base model and save
merged_model = model.merge_and_unload()
merged_model = merged_model.to(device)

In [ ]:
# Let's test the base model before training
prompt = "Salt and sugar can look very similar, but they taste very different. \
          Can you tell me how to distinguish between them?"

prompt = "Robots are becoming more and more common in our daily lives. \
Can you tell me how they are being used today?"

prompt = "Is it a good idea to go to the park?"

formatted_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}], tokenize=False
)
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

# Create a streamer
streamer = None

# streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
print("Before training:")

outputs = default_model.generate(**inputs, max_new_tokens=200, streamer=streamer, do_sample=True, temperature=0.3)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

print("\n\n ====== Training complete, generating new outputs ======")

print("After training:")
outputs = merged_model.generate(**inputs, max_new_tokens=300, streamer=streamer, do_sample=True, temperature=0.3)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

print("done")

Before training:
system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
user
Do you want milk or juice?
assistant
I'm glad you asked that. I'm a dairy product specialist, and I'm happy to help with any dairy product you're looking for. If you're looking for milk, I can provide you with a variety of options, from plain milk to flavored varieties like Greek yogurt and sour cream. If you're looking for juice, I can suggest a few options, such as apple juice or orange juice, and I can also recommend some specialty brands that offer unique flavors.

If you're looking for a quick and easy way to get a taste of dairy products, I can also suggest a few quick and easy recipes that include dairy products. For example, I can recommend a simple recipe for a homemade cheese or yogurt, or a quick and easy way to make a smoothie with almond milk or coconut milk.

What's your favorite dairy product to make?


 ====== Training complete, generating new outputs ======
After training: